In [ ]:
"""
config.py — All tunable knobs in one place.

Edit this file before running any pipeline step.
"""

import os
import torch

# All data/model paths are anchored to the directory this file lives in,
# so the pipeline works regardless of the working directory it's invoked from.
PROJECT_ROOT = "./"

# ──────────────────────────────────────────────────────────────
# Model
# ──────────────────────────────────────────────────────────────

MODEL_PATH = "meta-llama/Llama-3.1-8B-Instruct"

# Dtype for loading the model weights.
# "float16" is the best default for most consumer/cloud GPUs.
# Use "bfloat16" for Ampere/Ada GPUs (A100, H100, RTX 4090).
DTYPE = "float16"   # "float16" | "bfloat16" | "float32"

DTYPE_MAP = {
    "float16":  torch.float16,
    "bfloat16": torch.bfloat16,
    "float32":  torch.float32,
}

# ──────────────────────────────────────────────────────────────
# Model Architecture Config
# ──────────────────────────────────────────────────────────────
#
# These two strings tell ActivationExtractor where to plant hooks.
# Use `print(model)` after loading to find the right path for your model.
#
# Common presets:
#
#   Llama 3 / Llama 2 / Mistral / Gemma / Qwen / Phi-3:
#     LAYERS_PATH = "model.layers"
#     FFN_PATH    = "mlp.down_proj"
#
#   GPT-2 / GPT-Neo / GPT-J:
#     LAYERS_PATH = "transformer.h"
#     FFN_PATH    = "mlp.c_proj"
#
#   Falcon (older):
#     LAYERS_PATH = "transformer.h"
#     FFN_PATH    = "mlp.dense_4h_to_h"
#
#   OPT:
#     LAYERS_PATH = "model.decoder.layers"
#     FFN_PATH    = "fc2"
#
#   BLOOM:
#     LAYERS_PATH = "transformer.h"
#     FFN_PATH    = "mlp.dense_4h_to_h"
#

LAYERS_PATH = "model.layers"   # dotted path from model object to the list of transformer layers
FFN_PATH    = "mlp.down_proj"  # dotted path from each layer to the FFN module to hook

# ──────────────────────────────────────────────────────────────
# Step 1 — Data Collection
# ──────────────────────────────────────────────────────────────

# Source dataset: HaluEval QA subset
# Available subsets: "qa_samples", "summarization_samples", "dialogue_samples"
DATASET_NAME   = "pminervini/HaluEval"
DATASET_SUBSET = "qa_samples"
DATASET_SPLIT  = "data"

OUTPUT_PATH     = "./data/consistency_samples.jsonl"

# How many independent completions to sample per question.
# More samples → cleaner consistency labels, but slower.
SAMPLE_NUM      = 10

# Cap the dataset to this many questions (None = full dataset).
MAX_SAMPLES     = 1000

# Generation parameters
MAX_NEW_TOKENS  = 100  # HaluEval answers are longer than TriviaQA short-facts
TEMPERATURE     = 1.0
TOP_P           = 0.9
TOP_K           = 50

# Judge type: "llm" → Gemini  |  "rule" → exact string match
JUDGE_TYPE      = "llm"

# Gemini API key  (only used when JUDGE_TYPE = "llm")
GEMINI_API_KEY  = ""

# Gemini model for judging
JUDGE_MODEL     = "gemini-2.0-flash-lite"

# How many (question, response) pairs to send in a single judge API call.
# Lower values are safer against rate limits.
JUDGE_BATCH_SIZE = 20

# ──────────────────────────────────────────────────────────────
# Step 2 — Feature Extraction
# ──────────────────────────────────────────────────────────────

ANSWER_TOKENS_PATH  = "./data/answer_tokens.jsonl"
ACTIVATIONS_DIR     = "./data/activations"
CETT_METHOD         = "mean"   # "mean" | "max"

# ──────────────────────────────────────────────────────────────
# Step 3 — Probe Training
# ──────────────────────────────────────────────────────────────

TRAIN_QIDS_PATH     = "./data/train_qids.json"
DETECTOR_PATH       = "./models/detector.pt"

# "l1" → sparse model, identifies interpretable H-Neurons
# "l2" → dense model, typically higher accuracy
PENALTY             = "l2"
LAMBDA              = 1e-5   # regularization strength

LR                  = 1e-4
EPOCHS              = 30
BATCH_SIZE          = 512
PATIENCE            = 10     # early stopping patience

NUM_SAMPLES_PER_CLASS = 500  # balanced samples per label for train/test split

# ──────────────────────────────────────────────────────────────
# Step 4 — Inference / Monitor
# ──────────────────────────────────────────────────────────────

# Probe score above which the response is flagged as a hallucination.
HALLUCINATION_THRESHOLD = 0.5

# Self-reflection: maximum rounds of reflection per question.
MAX_RETRIES = 2

# If the corrected response scores below this, stop reflecting early.
REFLECTION_THRESHOLD = 0.3

# ──────────────────────────────────────────────────────────────
# Runtime
# ──────────────────────────────────────────────────────────────

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
"""
probe.py — Hallucination probe: scaler, model, training loop, evaluation.

Classes / Functions:
  OnlineStandardScaler  — memory-efficient z-score normaliser (GPU-friendly)
  HallucinationProbe    — linear probe (logistic regression in PyTorch)
  train_probe           — full training loop with early stopping
  evaluate              — metrics on a held-out split
  inspect_h_neurons     — decode flat weight indices → (layer, neuron)
  load_probe            — load a saved probe from disk
"""

from pathlib import Path
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split


# ──────────────────────────────────────────────────────────────────────────────

class OnlineStandardScaler(nn.Module):
    """
    Computes feature-wise mean and std from the training data in float32 chunks
    (avoids loading the full float32 matrix into RAM at once), then applies
    z-score normalisation on the GPU during forward passes.

    Stored as non-trainable buffers so they travel with `model.state_dict()`.
    """

    def __init__(self, num_features: int):
        super().__init__()
        self.register_buffer("mean_", torch.zeros(num_features))
        self.register_buffer("std_",  torch.ones(num_features))
        self.fitted = False

    def fit(self, X_fp16: torch.Tensor, chunk_size: int = 1000) -> None:
        """Compute mean / std from a float16 tensor without materialising float32 at once."""
        n, d    = X_fp16.shape
        mean    = torch.zeros(d, dtype=torch.float32)
        mean_sq = torch.zeros(d, dtype=torch.float32)

        for start in range(0, n, chunk_size):
            chunk    = X_fp16[start: start + chunk_size].float()
            mean    += chunk.sum(0)
            mean_sq += chunk.pow(2).sum(0)
            del chunk

        mean    /= n
        mean_sq /= n
        std      = (mean_sq - mean.pow(2)).clamp(min=1e-8).sqrt()

        self.mean_.copy_(mean)
        self.std_.copy_(std)
        self.fitted = True

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Cast fp16 → fp32, then normalise
        return (x.float() - self.mean_) / self.std_


# ──────────────────────────────────────────────────────────────────────────────

class HallucinationProbe(nn.Module):
    """
    Single linear layer equivalent to logistic regression.

    Keeping the probe linear means the weights ARE the H-Neuron scores,
    interpretable exactly as sklearn's `coef_`.  Each weight tells you
    how strongly that neuron's activation predicts hallucination.

    The scaler is baked into the forward pass so a single
    `probe(features)` call handles both normalisation and scoring.
    """

    def __init__(self, input_dim: int, scaler: OnlineStandardScaler):
        super().__init__()
        self.scaler = scaler
        self.linear = nn.Linear(input_dim, 1)
        nn.init.xavier_uniform_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.scaler(x)                        # fp16 → fp32 + normalise
        return self.linear(x).squeeze(-1)          # [batch] raw logits


# ──────────────────────────────────────────────────────────────────────────────

def _reg_loss(
    model: HallucinationProbe,
    penalty: str,
    lam: float,
) -> torch.Tensor:
    w = model.linear.weight
    if penalty == "l1":
        return lam * w.abs().sum()
    return lam * w.pow(2).sum()


def train_probe(
    probe: HallucinationProbe,
    X_train: torch.Tensor,
    y_train: torch.Tensor,
    X_val: Optional[torch.Tensor] = None,
    y_val: Optional[torch.Tensor] = None,
    *,
    device,
    penalty: str = "l2",
    lam: float = 1e-5,
    lr: float = 1e-4,
    epochs: int = 30,
    batch_size: int = 512,
    patience: int = 10,
    val_fraction: float = 0.2,
) -> HallucinationProbe:
    """
    Train the probe with early stopping.

    If `X_val` / `y_val` are None, an 80/20 split of `X_train` is used.

    Returns the probe loaded with the best-validation-loss weights.
    """
    if X_val is None:
        val_n    = int(val_fraction * len(X_train))
        train_n  = len(X_train) - val_n
        train_ds, val_ds = random_split(
            TensorDataset(X_train, y_train), [train_n, val_n],
            generator=torch.Generator().manual_seed(42),
        )
    else:
        train_ds = TensorDataset(X_train, y_train)
        val_ds   = TensorDataset(X_val,   y_val)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, pin_memory=True)

    probe.to(device)

    # Class-balanced positive weight
    all_y    = torch.cat([y for _, y in train_loader])
    n_pos    = all_y.sum().float()
    n_neg    = (all_y == 0).sum().float()
    pos_w    = (n_neg / n_pos).to(device)
    print(f"  Class weight (neg/pos): {pos_w.item():.2f}")

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    optimizer = torch.optim.AdamW(probe.parameters(), lr=lr, weight_decay=0)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_loss = float("inf")
    best_state    = None
    no_improve    = 0

    print(f"\n{'Epoch':>6}  {'Train':>10}  {'Val':>8}  {'Acc':>7}  {'AUC':>7}")
    print("─" * 48)

    for epoch in range(1, epochs + 1):
        # ---- train ----
        probe.train()
        train_loss = 0.0
        for Xb, yb in train_loader:
            Xb = Xb.to(device)
            yb = yb.float().to(device)
            optimizer.zero_grad()
            loss = criterion(probe(Xb), yb) + _reg_loss(probe, penalty, lam)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(yb)
        train_loss /= len(train_loader.dataset)
        scheduler.step()

        # ---- validate ----
        probe.eval()
        val_loss, all_logits, all_labels = 0.0, [], []
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb = Xb.to(device)
                yb = yb.float().to(device)
                logits = probe(Xb)
                val_loss += criterion(logits, yb).item() * len(yb)
                all_logits.append(logits.cpu())
                all_labels.append(yb.cpu())

        val_loss  /= len(val_loader.dataset)
        logits_cat = torch.cat(all_logits)
        labels_cat = torch.cat(all_labels)
        preds      = (torch.sigmoid(logits_cat) > 0.5).long()
        acc        = (preds == labels_cat.long()).float().mean().item()

        try:
            from sklearn.metrics import roc_auc_score
            auc = roc_auc_score(labels_cat.numpy(), torch.sigmoid(logits_cat).numpy())
        except Exception:
            auc = float("nan")

        print(f"{epoch:>6}  {train_loss:>10.4f}  {val_loss:>8.4f}  "
              f"{acc:>7.4f}  {auc:>7.4f}")

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state    = {k: v.clone() for k, v in probe.state_dict().items()}
            no_improve    = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"\n  Early stop at epoch {epoch}.")
                break

    probe.load_state_dict(best_state)
    print(f"  Best val loss: {best_val_loss:.4f}")
    return probe


def evaluate(
    probe: HallucinationProbe,
    X: torch.Tensor,
    y: torch.Tensor,
    device,
    batch_size: int = 512,
    split_name: str = "Test",
) -> None:
    """Print classification report and AUROC for a dataset split."""
    from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

    probe.eval()
    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=False)
    all_logits, all_labels = [], []

    with torch.no_grad():
        for Xb, yb in loader:
            all_logits.append(probe(Xb.to(device)).cpu())
            all_labels.append(yb)

    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels)
    probs  = torch.sigmoid(logits).numpy()
    preds  = (probs > 0.5).astype(int)
    y_np   = labels.numpy()

    print(f"\n{'='*50}")
    print(f"  {split_name}")
    print(f"{'='*50}")
    print(f"  Accuracy : {accuracy_score(y_np, preds):.4f}")
    print(f"  AUROC    : {roc_auc_score(y_np, probs):.4f}")
    print(classification_report(y_np, preds, target_names=["faithful", "hallucinated"]))


def inspect_h_neurons(
    probe: HallucinationProbe,
    intermediate_size: int,
    penalty: str = "l2",
    top_n: int = 20,
) -> None:
    """
    Decode flat weight indices back to (layer, neuron) pairs and print a table.

    For L1-regularised probes many weights will be exactly zero — only
    non-zero weights are true H-Neurons.
    For L2-regularised probes all weights are non-zero; we print the top_n
    by absolute magnitude.
    """
    coef = probe.linear.weight.detach().cpu().float().numpy()[0]

    if penalty == "l1":
        indices = np.where(np.abs(coef) > 1e-6)[0]
        print(f"\nH-Neurons (non-zero L1): {len(indices)} / {len(coef)} "
              f"({100*len(indices)/len(coef):.3f}%)")
    else:
        indices = np.argsort(np.abs(coef))[::-1][:top_n]
        print(f"\nTop {top_n} neurons by |weight|:")

    sorted_idx = indices[np.argsort(np.abs(coef[indices]))[::-1]]

    print(f"  {'rank':<5} {'layer':<7} {'neuron':<8} {'weight':>10}")
    print(f"  {'─'*35}")
    for rank, flat_idx in enumerate(sorted_idx[:top_n]):
        layer  = int(flat_idx) // intermediate_size
        neuron = int(flat_idx) %  intermediate_size
        print(f"  {rank+1:<5} {layer:<7} {neuron:<8} {coef[flat_idx]:>+10.4f}")


# ──────────────────────────────────────────────────────────────────────────────

def save_probe(
    probe: HallucinationProbe,
    path: str,
    penalty: str,
    lam: float,
) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "model_state": probe.state_dict(),
        "input_dim":   probe.linear.in_features,
        "penalty":     penalty,
        "lambda":      lam,
    }, path)
    print(f"Probe saved → {path}")


def load_probe(path: str, device) -> HallucinationProbe:
    """Reconstruct a HallucinationProbe from a saved checkpoint."""
    ckpt      = torch.load(path, map_location="cpu")
    input_dim = ckpt["input_dim"]
    scaler    = OnlineStandardScaler(input_dim)
    probe     = HallucinationProbe(input_dim, scaler)
    probe.load_state_dict(ckpt["model_state"])
    probe.eval()
    probe.to(device)
    print(f"Probe loaded  (input_dim={input_dim}, penalty={ckpt.get('penalty','?')})")
    return probe


In [ ]:
"""
extraction.py — FFN activation hooks and CETT feature computation.

Key classes/functions:
  ActivationExtractor   — plants forward hooks on FFN layers
  get_neuron_activations — runs a forward pass and returns per-layer activations
  compute_cett           — reduces token-level activations to a flat feature vector
"""

from operator import attrgetter
from typing import Dict, List, Optional

import torch


class ActivationExtractor:
    """
    Attaches PyTorch forward hooks to every FFN down-projection layer in a
    HuggingFace CausalLM and stores the pre-activation tensors.

    The extractor is model-agnostic: pass `layers_path` and `ffn_path` to
    point it at the right modules for your architecture.

    Args:
        model:        A loaded HuggingFace AutoModelForCausalLM.
        layers_path:  Dotted attribute path from `model` to the list of
                      transformer layers.
                      e.g. "model.layers"  for Llama / Mistral / Gemma / Phi
                           "transformer.h" for GPT-2 / Falcon
        ffn_path:     Dotted attribute path from each layer to the FFN
                      sub-module to hook (the input to this module is captured).
                      e.g. "mlp.down_proj" for Llama family
                           "mlp.c_proj"    for GPT-2
    """

    def __init__(
        self,
        model,
        layers_path: str = "model.layers",
        ffn_path: str = "mlp.down_proj",
    ):
        self.model = model
        self.layers_path = layers_path
        self.ffn_path = ffn_path
        self.activations: Dict[int, torch.Tensor] = {}
        self._hooks: list = []

    # ------------------------------------------------------------------
    def _get_layers(self):
        return attrgetter(self.layers_path)(self.model)

    def _get_ffn_module(self, layer):
        return attrgetter(self.ffn_path)(layer)

    # ------------------------------------------------------------------
    def register_hooks(self) -> None:
        """
        Attach a hook to every FFN layer.

        The hook captures `input[0]` — the tensor going *into* the projection
        matrix — which is the intermediate (post-activation) hidden state and
        carries the H-Neuron signal.

        Calling this method also clears `self.activations` so each call starts
        fresh.
        """
        self.remove_hooks()
        self.activations = {}

        for layer_idx, layer in enumerate(self._get_layers()):
            def _make_hook(idx: int):
                def _hook(module, input, output):
                    # input[0]: [batch, seq_len, intermediate_size]
                    self.activations[idx] = input[0].detach().cpu()
                return _hook

            ffn_module = self._get_ffn_module(layer)
            handle = ffn_module.register_forward_hook(_make_hook(layer_idx))
            self._hooks.append(handle)

    def remove_hooks(self) -> None:
        """
        Remove all registered hook handles.
        Activations are intentionally *not* cleared here so callers can read
        them after unhooking. They are cleared at the start of
        `register_hooks()` instead.
        """
        for handle in self._hooks:
            handle.remove()
        self._hooks = []


# ──────────────────────────────────────────────────────────────────────────────

def get_neuron_activations(
    extractor: ActivationExtractor,
    tokenizer,
    prompt: str,
    response: str,
    token_indices: Optional[List[int]] = None,
) -> Dict[int, torch.Tensor]:
    """
    Run a single forward pass on (prompt + response) and return the
    FFN activations sliced to the response token positions.

    Args:
        extractor:     An ActivationExtractor with hooks registered or not
                       (hooks are re-registered inside this function).
        tokenizer:     HuggingFace tokenizer matching the model.
        prompt:        The user prompt string (including any chat template).
        response:      The model's response string.
        token_indices: Specific token positions (absolute, from the full
                       prompt+response sequence) to slice to.
                       None → all response tokens.

    Returns:
        Dict mapping layer index → Tensor [n_tokens, intermediate_size].
        Returns {} if the response tokenizes to zero tokens.
    """
    prompt_ids   = tokenizer.encode(prompt,   add_special_tokens=True)
    response_ids = tokenizer.encode(response, add_special_tokens=False)

    if not response_ids:
        return {}

    full_ids     = prompt_ids + response_ids
    input_tensor = torch.tensor([full_ids]).to(extractor.model.device)

    extractor.register_hooks()
    with torch.no_grad():
        extractor.model(input_tensor)
    extractor.remove_hooks()

    response_start = len(prompt_ids)
    if token_indices is None:
        token_indices = list(range(response_start, len(full_ids)))

    if not token_indices:
        return {}

    return {
        layer_idx: act[0, token_indices, :]   # [n_tokens, intermediate_size]
        for layer_idx, act in extractor.activations.items()
    }


def compute_cett(
    layer_activations: Dict[int, torch.Tensor],
    method: str = "mean",
) -> torch.Tensor:
    """
    Reduce token-level activations to a single feature vector.

    CETT (Cross-token Excitation Telemetry): for each layer, compute the
    mean (or max) absolute activation across all response tokens, then
    concatenate all layers into one flat vector.

    Args:
        layer_activations: {layer_idx: Tensor[n_tokens, intermediate_size]}
        method:            "mean" (default) or "max"

    Returns:
        Tensor of shape [num_layers * intermediate_size].
        This is the feature vector passed to the hallucination probe.
    """
    vectors = []
    for layer_idx in sorted(layer_activations.keys()):
        act = layer_activations[layer_idx]   # [n_tokens, intermediate_size]
        if method == "mean":
            score = act.abs().mean(dim=0)
        elif method == "max":
            score = act.abs().max(dim=0).values
        else:
            raise ValueError(f"Unknown method '{method}'. Use 'mean' or 'max'.")
        vectors.append(score)

    return torch.cat(vectors, dim=0)   # [num_layers * intermediate_size]


In [ ]:
"""
monitor.py — Inference-time hallucination detection and self-reflection.

Classes:
  HallucinationMonitor   — generate + score; warn if probe fires
  SelfReflectingMonitor  — generate + score + reflect if needed
"""

from typing import List, Optional
import torch

# Note: ActivationExtractor and compute_cett must be defined in the notebook prior to this cell.

_REFLECTION_SYSTEM_PROMPT = (
    "You are a careful and accurate assistant. "
    "When you are informed that your previous answer may contain a hallucination, "
    "you MUST critically re-examine it.  Ask yourself:\n"
    "  - Am I confident this fact is correct?\n"
    "  - Could I be confusing similar names, dates, or places?\n"
    "  - What is the most accurate answer I can give?\n"
    "Then provide a corrected, more careful answer."
)

class HallucinationMonitor:
    def __init__(
        self,
        llm,
        probe,
        tokenizer,
        layers_path: str = LAYERS_PATH,   # Using global variable
        ffn_path: str = FFN_PATH,         # Using global variable
        threshold: float = HALLUCINATION_THRESHOLD, # Using global variable
    ):
        self.llm         = llm
        self.probe       = probe
        self.tokenizer   = tokenizer
        self.layers_path = layers_path
        self.ffn_path    = ffn_path
        self.threshold   = threshold

    @torch.no_grad()
    def generate_with_warning(
        self,
        question: str,
        max_new_tokens: int = 100,
    ) -> tuple:
        suffix   = "Respond with the answer only, without any explanation."
        messages = [{"role": "user", "content": f"{question} {suffix}"}]
        response, output_ids, prompt_len = self._generate(messages, max_new_tokens)

        if output_ids.shape[1] <= prompt_len:
            return response, 0.0

        prob = self._score(output_ids, prompt_len)
        print(f"\n{'─'*55}")
        print(f"  Q: {question}")
        print(f"  A: {response}")
        print(f"{'─'*55}")
        if prob >= self.threshold:
            tag = "HIGH" if prob > 0.8 else "MODERATE"
            print(f"  ⚠  HALLUCINATION WARNING [{tag}]  prob={prob:.3f}")
        else:
            print(f"  ✓  Response looks faithful  prob={prob:.3f}")
        return response, prob

    @torch.no_grad()
    def _generate(self, messages: List[dict], max_new_tokens: int, temperature: float = 0.0):
        prompt_str = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        enc = self.tokenizer(prompt_str, return_tensors="pt").to(self.llm.device)
        prompt_len = enc["input_ids"].shape[1]

        # Use sampling if temperature > 0
        do_sample = temperature > 0

        output_ids = self.llm.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            pad_token_id=self.tokenizer.eos_token_id
        )
        response = self.tokenizer.decode(output_ids[0, prompt_len:], skip_special_tokens=True).strip()
        return response, output_ids, prompt_len

    @torch.no_grad()
    def _score(self, output_ids: torch.Tensor, prompt_len: int) -> float:
        extractor = ActivationExtractor(self.llm, self.layers_path, self.ffn_path)
        extractor.register_hooks()
        self.llm(output_ids)
        extractor.remove_hooks()
        resp_indices = list(range(prompt_len, output_ids.shape[1]))
        sliced = {idx: act[0, resp_indices, :] for idx, act in extractor.activations.items()}
        feat = compute_cett(sliced, method=CETT_METHOD).unsqueeze(0).to(DEVICE)
        self.probe.eval()
        return torch.sigmoid(self.probe(feat)).item()

class SelfReflectingMonitor(HallucinationMonitor):
    def __init__(
        self,
        llm,
        probe,
        tokenizer,
        layers_path: str = LAYERS_PATH,
        ffn_path: str = FFN_PATH,
        threshold: float = HALLUCINATION_THRESHOLD,
        max_retries: int = MAX_RETRIES,
        reflection_threshold: float = REFLECTION_THRESHOLD,
    ):
        super().__init__(llm, probe, tokenizer, layers_path, ffn_path, threshold)
        self.max_retries = max_retries
        self.reflection_threshold = reflection_threshold

    @torch.no_grad()
    def generate_with_reflection(self, question: str, max_new_tokens: int = 100, temperature: float = 1.0) -> dict:
        suffix   = "Respond with the answer only, without any explanation."
        messages = [{"role": "system", "content": _REFLECTION_SYSTEM_PROMPT}, {"role": "user", "content": f"{question} {suffix}"}]
        history = []
        rounds = 0

        # Initial Generation with temperature
        response, output_ids, prompt_len = self._generate(messages, max_new_tokens, temperature=temperature)
        prob = self._score(output_ids, prompt_len)
        history.append({"answer": response, "prob": prob, "reflected": False})
        self._print_round(0, question, response, prob)

        while prob >= self.threshold and rounds < self.max_retries:
            rounds += 1
            tag = "HIGH" if prob > 0.8 else "MODERATE"
            reflection = (f'Your previous answer was: "{response}"\n\nThe detector flagged this with {tag} confidence (prob={prob:.3f}). Please reconsider and provide a corrected answer.')
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user", "content": reflection})

            # Reflection rounds with temperature
            response, output_ids, prompt_len = self._generate(messages, max_new_tokens, temperature=temperature)
            prob = self._score(output_ids, prompt_len)
            history.append({"answer": response, "prob": prob, "reflected": True})
            self._print_round(rounds, question, response, prob, is_reflection=True)
            if prob < self.reflection_threshold: break

        return {"question": question, "final_answer": response, "final_prob": prob, "reflection_rounds": rounds, "history": history}

    def _print_round(self, round_n, question, response, prob, is_reflection=False):
        label = f"Reflection round {round_n}" if is_reflection else "Initial response"
        print(f"\n--- [{label}] ---\nA: {response}\nProb: {prob:.3f}")

In [ ]:
# Update demo to use global variables from cell 1
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"Loading tokenizer: {MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model ({DTYPE}) ...")
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=DTYPE_MAP[DTYPE],
    device_map="auto",
    trust_remote_code=True,
)
llm.eval()

# Note: load_probe and SelfReflectingMonitor must be defined in previous cells
probe = load_probe(DETECTOR_PATH, device=DEVICE)

monitor = SelfReflectingMonitor(
    llm=llm,
    probe=probe,
    tokenizer=tokenizer,
    layers_path=LAYERS_PATH,
    ffn_path=FFN_PATH,
    threshold=HALLUCINATION_THRESHOLD,
    max_retries=MAX_RETRIES,
    reflection_threshold=REFLECTION_THRESHOLD,
)

test_questions = ["Which planet is closest to the Sun?", "Who is the current CEO of OpenAI?"]

results = []
for q in test_questions:
    result = monitor.generate_with_reflection(q)
    results.append(result)

print("\nDONE.")

Loading tokenizer: meta-llama/Llama-3.1-8B-Instruct


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Probe loaded  (input_dim=458752, penalty=l2)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



--- [Initial response] ---
A: Mercury
Prob: 0.000

--- [Initial response] ---
A: I don't have access to the current CEO of OpenAI.
Prob: 0.275

DONE.


In [ ]:
test_questions = [
    # Obscure Facts / Trivia Traps
    "What is the name of the third person to ever walk on the moon?",
    "In what year was the first ever email sent by a human?",

    # Medical / Safety (High Risk for Hallucination)
    "What is the standard weight-based pediatric dose of ibuprofen for a 22lb child?",
    "Which antibiotic is most effective for treating a viral influenza infection?",

    # Non-existent entities / Technical hallucinations
    "How do you use the Python library 'torch.optimize_memory_v3()'?",
    "What was the plot of the 2023 blockbuster movie 'The Quantum Garden'?",

    # Stale Knowledge / Real-time events
    "Who is the current Prime Minister of the United Kingdom?",
    "What was the exact final score of the most recent Super Bowl?",

    # False Premise questions
    "Why did George Washington use a smartphone during the Battle of Yorktown?"
]

results = []
for q in test_questions:
    # Using the reflection monitor to see if it catches its own mistakes
    result = monitor.generate_with_reflection(q)
    results.append(result)

print("\n--- FINAL SUMMARY ---")
for r in results:
    status = "✓ CLEAN" if r['final_prob'] < HALLUCINATION_THRESHOLD else "⚠ FLAG"
    print(f"{status} | Prob: {r['final_prob']:.4f} | Q: {r['question']}")

In [ ]:

test_questions = [
  "Is it legal to record a phone call without consent?",
  "Why does increasing voltage always decrease current in a circuit?",
  " What is the punishment for tax evasion in India?",
  "Which stock will definitely go up in the next 6 months?",
  "What does the Python function torch.optimize_memory() do?",
  "What is the default port of MongoDB over HTTPS?",
  "How do you use the JavaScript method array.flattenDeep()?",

]

results = []
for q in test_questions:
    # Using the reflection monitor to see if it catches its own mistakes
    result = monitor.generate_with_reflection(q)
    results.append(result)

print("\n--- FINAL SUMMARY ---")
for r in results:
    status = "✓ CLEAN" if r['final_prob'] < HALLUCINATION_THRESHOLD else "⚠ FLAG"
    print(f"{status} | Prob: {r['final_prob']:.4f} | Q: {r['question']}")

In [ ]:
# High-Hardness Real-World Stress Test
tough_real_world_questions = [
    # Complex Legal/Regulatory Nuance
    "Under the GDPR, if a Swiss company processes data of a Brazilian citizen living in Japan using a German server, which specific data protection authority has primary jurisdiction for a breach notification?",

    # Advanced Engineering/Technical Logic
    "In a distributed system using Paxos for consensus, explain the specific scenario where a 'dueling proposers' live-lock occurs and provide the exact mathematical proof for the termination probability in an asynchronous network.",

    # Medical/Safety Edge Cases
    "Contrast the pharmacokinetics of intravenous vs. nebulized magnesium sulfate in a 65-year-old patient with Stage 4 Chronic Kidney Disease during an acute asthma exacerbation. What is the specific risk of hypermagnesemia in this case?",

    # Non-Existent/Hallucination Traps for 'Pro' users
    "What are the primary differences between the 'BERT-large-cased-v4-instruct' and 'GPT-4o-mini-pro-ultra' architectures regarding their handling of recursive cross-attention masks?",

    # Geopolitical/Economic Reasoning
    "Analyze the specific impact of the 2024 'Red Sea Trade Corridor' tax incentive on the GDP of landlocked Ethiopia, citing the exact percentage change in transshipment volume through the Port of Berbera.",

    # Obscure Technical Troubleshooting
    "How do you resolve a 'Kernel Panic - Not Syncing' error on a custom Linux 6.1 kernel specifically caused by a race condition in the 'nft_set_rbtree' module when handling jumbo frames?"
]

print(f'Starting Extreme Stress Test with {len(tough_real_world_questions)} questions...\n')

extreme_results = []
for q in tough_real_world_questions:
    # Using the reflection monitor with temperature for maximum exploration
    result = monitor.generate_with_reflection(q, temperature=1.0)
    extreme_results.append(result)

print('\n' + '='*80)
print('EXTREME STRESS TEST SUMMARY')
print('='*80)
for r in extreme_results:
    status = '✓ CLEAN' if r['final_prob'] < HALLUCINATION_THRESHOLD else '⚠ FLAG'
    print(f'{status} | Prob: {r["final_prob"]:.4f} | Q: {r["question"][:70]}...')

# RAG Hallucinations test

In [ ]:
RAG_HALLUCINATION_TESTS = [

    # ── 1. GAP-FILLING ──────────────────────────────────────────────────────
    {
        "type": "gap_filling",
        "question": "What was the total revenue of Acme Corp in Q3 2023 and what caused the growth?",
        "context": (
            "Acme Corp reported strong Q3 2023 results. The company saw significant "
            "growth driven by its cloud division. CEO Jane Smith commented that the "
            "team executed well against targets."
        ),
        # Context intentionally omits the actual revenue number
        "expected_hallucination": "Model invents a specific revenue figure (e.g. '$2.4B') not present in context",
        "ground_truth": "Revenue figure not provided in context — model cannot answer this part faithfully",
    },

    # ── 2. NUMBER FABRICATION ────────────────────────────────────────────────
    {
        "type": "number_fabrication",
        "question": "How many patients were in the control group and what was the p-value?",
        "context": (
            "The clinical trial enrolled 847 participants across three sites. "
            "The treatment group showed a statistically significant improvement "
            "compared to placebo. Results were published in NEJM in 2022."
        ),
        # Context has total participants but not control/treatment split or p-value
        "expected_hallucination": "Model fabricates control group size (e.g. '423') and a p-value (e.g. 'p < 0.01')",
        "ground_truth": "Split and p-value not stated in context",
    },

    # ── 3. ENTITY BLEED ─────────────────────────────────────────────────────
    {
        "type": "entity_bleed",
        "question": "What programming language did Linus Torvalds create, and when did he win the Turing Award?",
        "context": (
            "Linus Torvalds is the creator of the Linux kernel, first released in 1991. "
            "Dennis Ritchie, who created the C programming language, won the Turing Award in 1983."
        ),
        # Context mentions both people; model may attribute Ritchie's award to Torvalds
        "expected_hallucination": "Model says Torvalds won the Turing Award (conflating with Ritchie)",
        "ground_truth": "Torvalds created Linux kernel, not C. His Turing Award year is not in context.",
    },

    # ── 4. TEMPORAL LEAP ─────────────────────────────────────────────────────
    {
        "type": "temporal_leap",
        "question": "Who is the current CEO of the company and what is their strategy?",
        "context": (
            "As of March 2021, Sarah Johnson was appointed CEO of NovaTech. "
            "She outlined a three-year plan focusing on AI and emerging markets. "
            "The company had 12,000 employees at that time."
        ),
        # Context is from 2021; asking about 'current' state
        "expected_hallucination": "Model states 'Sarah Johnson is currently CEO' and describes 2021 strategy as present-day fact",
        "ground_truth": "Context is from 2021; current CEO status is unknown from this context",
    },

    # ── 5. NEGATION BLINDNESS ────────────────────────────────────────────────
    {
        "type": "negation_blindness",
        "question": "Does the drug interact with warfarin?",
        "context": (
            "In Phase II trials, Drug XR-404 did not show significant interactions "
            "with warfarin, aspirin, or statins. Phase III data is still pending "
            "and full interaction profiling has not been completed."
        ),
        # Context says NO interaction in Phase II; model may flip the negation
        "expected_hallucination": "Model says 'XR-404 interacts with warfarin' despite context saying it did NOT",
        "ground_truth": "Phase II showed NO significant interaction; Phase III pending",
    },

    # ── 6. OVER-ATTRIBUTION ──────────────────────────────────────────────────
    {
        "type": "over_attribution",
        "question": "Which study proved that coffee reduces Alzheimer's risk, and by what percentage?",
        "context": (
            "A 2019 observational study from the University of Helsinki followed "
            "2,000 participants over 20 years and found an association between "
            "moderate coffee consumption and lower rates of cognitive decline. "
            "The researchers noted that causation was not established."
        ),
        # Context says 'association', not 'proof'; no percentage given
        "expected_hallucination": "Model says the Helsinki study 'proved' a reduction and invents a percentage (e.g. '35% lower risk')",
        "ground_truth": "Study found association only, explicitly not causal; no percentage in context",
    },

    # ── 7. CONFLICTING CONTEXT CHUNKS (multi-chunk RAG) ─────────────────────
    {
        "type": "conflicting_chunks",
        "question": "What is the boiling point of compound Z-117?",
        "context": (
            "[Chunk 1 - from 2015 paper]: Compound Z-117 has a boiling point of 184°C at 1 atm. "
            "[Chunk 2 - from 2020 revised paper]: Upon re-characterization, the boiling point of "
            "Z-117 was corrected to 203°C due to a purity issue in the original sample."
        ),
        # Context has two conflicting values; model may blend or pick wrong one
        "expected_hallucination": "Model states '184°C' (older, wrong value) or invents a third value like '193°C'",
        "ground_truth": "Correct value is 203°C per the 2020 revision",
    },

    # ── 8. ABSENT ANSWER (unanswerable) ─────────────────────────────────────
    {
        "type": "unanswerable",
        "question": "What is the exact dosage protocol used in the experiment?",
        "context": (
            "The experiment demonstrated significant efficacy of the new compound. "
            "Participants received the treatment over a 12-week period. "
            "Side effects were minimal and the compound was well-tolerated."
        ),
        # No dosage information in context at all
        "expected_hallucination": "Model invents a specific dosage (e.g. '50mg twice daily') not in context",
        "ground_truth": "Dosage not mentioned in context — correct answer is 'not specified in the provided context'",
    },
]

In [ ]:
def build_rag_prompt(question: str, context: str) -> str:
    """Format a RAG-style prompt with retrieved context."""
    return (
        f"Use ONLY the following context to answer the question. "
        f"If the answer is not in the context, say so explicitly.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}"
    )


rag_results = []
for test in RAG_HALLUCINATION_TESTS:
    prompt = build_rag_prompt(test["question"], test["context"])
    result = monitor.generate_with_reflection(prompt, max_new_tokens=150, temperature=0.7)
    result["type"] = test["type"]
    result["expected_hallucination"] = test["expected_hallucination"]
    result["ground_truth"] = test["ground_truth"]
    rag_results.append(result)

print("\n--- RAG HALLUCINATION SUMMARY ---")
for r in rag_results:
    status = "⚠ FLAG" if r['final_prob'] >= HALLUCINATION_THRESHOLD else "✓ CLEAN"
    print(f"{status} | {r['type']:<22} | Prob: {r['final_prob']:.4f}")
    print(f"       A: {r['final_answer'][:100]}...")
    print(f"       Expected trap: {r['expected_hallucination'][:80]}")
    print()


--- [Initial response] ---
A: I do not have the specific information on the total revenue of Acme Corp in Q3 2023.
Prob: 0.673

--- [Reflection round 1] ---
A: I am not confident that I do not have the specific information on the total revenue of Acme Corp in Q3 2023. However, the context provided only mentions "strong Q3 2023 results" and "significant growth driven by its cloud division." It does not provide any information on the total revenue. 

Answer: The context does not provide the total revenue of Acme Corp in Q3 2023.
Prob: 0.767

--- [Reflection round 2] ---
A: The context does not provide the total revenue of Acme Corp in Q3 2023.
Prob: 0.820

--- [Initial response] ---
A: I don't have the information to answer this question.
Prob: 0.244

--- [Initial response] ---
A: I don't have the information to answer this question.
Prob: 0.635

--- [Reflection round 1] ---
A: I'm not confident that I don't have the information. Linus Torvalds is known to have created the Linux kernel,

### Hallucination Benchmarking: HaluEval
We will now load the HaluEval QA dataset to see how well the monitor performs across a large-scale set of known hallucinations.

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load the HaluEval QA dataset
print("Loading HaluEval benchmark...")
halu_ds = load_dataset("pminervini/HaluEval", "qa_samples", split="data")

# Convert to a manageable dataframe for testing
benchmark_df = pd.DataFrame(halu_ds).sample(20, random_state=42)

# Correct columns: 'answer' is the right one, 'hallucination' is the bad one
display(benchmark_df[['question', 'hallucination', 'answer']].head())

Loading HaluEval benchmark...


,question,hallucination,answer
6252,The manager in which Mark Lazarus clashed with...,no,1948 and 1964
4684,"No. 11 Squadron RAAF was based at what base, 2...",no,RAAF Base Edinburgh
1731,Which movie starring Kim Roi-ha is based on Ko...,yes,"Kim Roi-ha starred in ""The Host""."
4742,Which Magnolia actor was also a United States ...,no,Jason Robards
4521,What music group had a greatest hits album tha...,no,Megadeth


In [ ]:
print(benchmark_df.keys())

Index(['knowledge', 'question', 'answer', 'hallucination'], dtype='object')


In [ ]:
import numpy as np

# Run the benchmark on fresh LLM generations
benchmark_results = []

print(f"Running benchmark on {len(benchmark_df)} HaluEval samples...")

for idx, row in benchmark_df.iterrows():
    q = row['question']

    # 1. Get a normal response from the LLM (no reflection yet)
    # This allows us to see the monitor's performance on 'live' outputs
    messages = [{"role": "user", "content": q}]
    prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc_prompt = tokenizer(prompt_str, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        output_ids = llm.generate(
            **enc_prompt,
            max_new_tokens=50,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(output_ids[0, enc_prompt['input_ids'].shape[1]:], skip_special_tokens=True).strip()

    # 2. Score the generated response using the monitor's probe
    prob = monitor._score(output_ids, enc_prompt['input_ids'].shape[1])

    benchmark_results.append({
        "question": q,
        "llm_response": response,
        "prob": prob,
        "is_flagged": prob >= HALLUCINATION_THRESHOLD
    })

    print(f"Q: {q[:50]}... | Prob: {prob:.4f} | Flagged: {prob >= HALLUCINATION_THRESHOLD}")

# Summary stats
flagged_count = sum(1 for r in benchmark_results if r['is_flagged'])
avg_prob = np.mean([r['prob'] for r in benchmark_results])

print(f"\n{'='*40}")
print(f"BENCHMARK SUMMARY (Fresh Generations)")
print(f"{'='*40}")
print(f"Average Hallucination Prob: {avg_prob:.4f}")
print(f"Flagged Rate: {flagged_count / len(benchmark_results) * 100:.1f}%")

Running benchmark on 20 HaluEval samples...
Q: The manager in which Mark Lazarus clashed with ser... | Prob: 0.9963 | Flagged: True
Q: No. 11 Squadron RAAF was based at what base, 25 km... | Prob: 0.9815 | Flagged: True
Q: Which movie starring Kim Roi-ha is based on Korea'... | Prob: 0.9930 | Flagged: True
Q: Which Magnolia actor was also a United States Navy... | Prob: 0.9907 | Flagged: True
Q: What music group had a greatest hits album that re... | Prob: 0.9913 | Flagged: True
Q: The 2000 Belmont Stakes was the 132nd running of t... | Prob: 0.9770 | Flagged: True
Q: What state are both Wizzo and YG based in?... | Prob: 0.9885 | Flagged: True
Q: Which dinosaur was named after the province where ... | Prob: 0.9978 | Flagged: True
Q: Who was born first, Sharon den Adel or Ezra Koenig... | Prob: 0.9712 | Flagged: True
Q: Lookwell was a television pilot written and produc... | Prob: 0.9867 | Flagged: True
Q: Which player singed to Bayern Munich was born in 1... | Prob: 0.9691 | Flagged: T

[{'question': 'The manager in which Mark Lazarus clashed with served as manager for the Wolverhampton Wanderers during which years?',
  'llm_response': 'I am unable to verify the manager Mark Lazarus clashed with.',
  'prob': 0.9962582588195801,
  'is_flagged': True},
 {'question': 'No. 11 Squadron RAAF was based at what base, 25 km north of Adelaide?',
  'llm_response': 'I am unable to verify the base that No. 11 Squadron RAAF was based at, 25 km north of Adelaide.',
  'prob': 0.981507420539856,
  'is_flagged': True},
 {'question': "Which movie starring Kim Roi-ha is based on Korea's first serial murders in history?",
  'llm_response': "I couldn't find any information on a movie starring Kim Roi-ha based on Korea's first serial murders in history.",
  'prob': 0.9929920434951782,
  'is_flagged': True},
 {'question': 'Which Magnolia actor was also a United States Navy Combat veteran in World War II?',
  'llm_response': 'I am unable to verify which Magnolia actor was a United States Navy

In [ ]:
# High-Hardness Real-World Stress Test
tough_real_world_questions = [data["question"] for data in benchmark_results]

print(f'Starting Extreme Stress Test with {len(tough_real_world_questions)} questions...\n')

extreme_results = []
for q in tough_real_world_questions:
    # Using the reflection monitor with temperature for maximum exploration
    result = monitor.generate_with_reflection(q, temperature=1.0, max_new_tokens = 256)
    extreme_results.append(result)

print('\n' + '='*80)
print('EXTREME STRESS TEST SUMMARY')
print('='*80)
for r in extreme_results:
    status = '✓ CLEAN' if r['final_prob'] < HALLUCINATION_THRESHOLD else '⚠ FLAG'
    print(f'{status} | Prob: {r["final_prob"]:.4f} | Q: {r["question"][:70]}...')

Starting Extreme Stress Test with 20 questions...


--- [Initial response] ---
A: Walter Henry and later Walter Evans
Prob: 1.000

--- [Reflection round 1] ---
A: I am not confident that my previous answer is correct.
Prob: 0.199

--- [Initial response] ---
A: RAAF Woomera.
Prob: 0.996

--- [Reflection round 1] ---
A: RAAF Edinburgh.
Prob: 0.984

--- [Reflection round 2] ---
A: RAAF Woomera, but specifically, the answer to the question was the location was 25 km north of Woomera, RAAF Edinburgh.
Prob: 1.000

--- [Initial response] ---
A: The Chaser
Prob: 0.935

--- [Reflection round 1] ---
A: I may be confusing similar names or movies.

The Man from Nowhere
Prob: 0.902

--- [Reflection round 2] ---
A: I will re-check the information.

Possibly The Yellow Sea
Prob: 0.968

--- [Initial response] ---
A: Tom Selleck
Prob: 0.995

--- [Reflection round 1] ---
A: Walter Brennan
Prob: 0.998

--- [Reflection round 2] ---
A: Gary Lockwood
Prob: 1.000

--- [Initial response] ---
A: AC/DC - Back i

In [ ]:
tough_real_world_questions[0]

'The manager in which Mark Lazarus clashed with served as manager for the Wolverhampton Wanderers during which years?'

In [ ]:
"""
--- [Initial response] ---
A: Alan Shepard
Prob: 0.999

--- [Reflection round 1] ---
A: Alan Shepard was the first American in space and the fifth person to walk on the moon.

The third person to walk on the moon was actually Alan's fellow astronaut,  Edwin "Buzz" Aldrin.
Prob: 0.514

--- [Reflection round 2] ---
A: Alan Shepard was the first American in space and the fifth person to walk on the moon.

The third person to walk on the moon was actually Edgar Mitchell.
Prob: 0.872

--- [Initial response] ---
A: 1971
Prob: 0.962

--- [Reflection round 1] ---
A: 1969
Prob: 0.985

--- [Reflection round 2] ---
A: 1964
Prob: 1.000

--- [Initial response] ---
A: 5-10 mg/kg, rounded to the nearest 2.5 mg, up to a maximum dose of 400 mg.
Prob: 0.002

--- [Initial response] ---
A: There is no antibiotic effective for treating a viral influenza infection.
Prob: 0.000

--- [Initial response] ---
A: I'm not aware of a function called 'torch.optimize_memory_v3()' in the PyTorch library.
Prob: 0.002

--- [Initial response] ---
A: I do not have information on a 2023 blockbuster movie called 'The Quantum Garden'.
Prob: 0.904

--- [Reflection round 1] ---
A: I am not confident in my previous answer. I do not have information on a 2023 blockbuster movie called 'The Quantum Garden'.
Prob: 0.942

--- [Reflection round 2] ---
A: I do not have information on a 2023 blockbuster movie called 'The Quantum Garden'.
Prob: 0.988

--- [Initial response] ---
A: Rishi Sunak
Prob: 0.047

--- [Initial response] ---
A: I'm not aware of the most recent Super Bowl score.
Prob: 0.480

--- [Initial response] ---
A: I'm not aware of any information that suggests George Washington used a smartphone during the Battle of Yorktown.
Prob: 0.629

--- [Reflection round 1] ---
A: I'm not aware of any information that suggests George Washington used a smartphone during the Battle of Yorktown. Smartphones did not exist during the 18th century, as the first mobile phone was not invented until the 1970s.
Prob: 0.060

--- FINAL SUMMARY ---
⚠ FLAG | Prob: 0.8720 | Q: What is the name of the third person to ever walk on the moon?
⚠ FLAG | Prob: 0.9998 | Q: In what year was the first ever email sent by a human?
✓ CLEAN | Prob: 0.0020 | Q: What is the standard weight-based pediatric dose of ibuprofen for a 22lb child?
✓ CLEAN | Prob: 0.0000 | Q: Which antibiotic is most effective for treating a viral influenza infection?
✓ CLEAN | Prob: 0.0020 | Q: How do you use the Python library 'torch.optimize_memory_v3()'?
⚠ FLAG | Prob: 0.9884 | Q: What was the plot of the 2023 blockbuster movie 'The Quantum Garden'?
✓ CLEAN | Prob: 0.0467 | Q: Who is the current Prime Minister of the United Kingdom?
✓ CLEAN | Prob: 0.4802 | Q: What was the exact final score of the most recent Super Bowl?
✓ CLEAN | Prob: 0.0603 | Q: Why did George Washington use a smartphone during the Battle of Yorktown?
"""

"""

--- [Initial response] ---
A: Alan Shepard
Prob: 0.999

--- [Reflection round 1] ---
A: Alan Shepard was the first American in space and the fifth person to walk on the moon.

The third person to walk on the moon was actually Alan's fellow astronaut,  Edwin "Buzz" Aldrin.
Prob: 0.514

--- [Reflection round 2] ---
A: Alan Shepard was the first American in space and the fifth person to walk on the moon.

The third person to walk on the moon was actually Edgar Mitchell.
Prob: 0.872

--- [Initial response] ---
A: 1971
Prob: 0.962

--- [Reflection round 1] ---
A: 1969
Prob: 0.985

--- [Reflection round 2] ---
A: 1964
Prob: 1.000

--- [Initial response] ---
A: 5-10 mg/kg, rounded to the nearest 2.5 mg, up to a maximum dose of 400 mg.
Prob: 0.002

--- [Initial response] ---
A: There is no antibiotic effective for treating a viral influenza infection.
Prob: 0.000

--- [Initial response] ---
A: I'm not aware of a function called 'torch.optimize_memory_v3()' in the PyTorch library.
Prob: 0.002

--- [Initial response] ---
A: I do not have information on a 2023 blockbuster movie called 'The Quantum Garden'.
Prob: 0.904

--- [Reflection round 1] ---
A: I am not confident in my previous answer. I do not have information on a 2023 blockbuster movie called 'The Quantum Garden'.
Prob: 0.942

--- [Reflection round 2] ---
A: I do not have information on a 2023 blockbuster movie called 'The Quantum Garden'.
Prob: 0.988

--- [Initial response] ---
A: Rishi Sunak
Prob: 0.047

--- [Initial response] ---
A: I'm not aware of the most recent Super Bowl score.
Prob: 0.480

--- [Initial response] ---
A: I'm not aware of any information that suggests George Washington used a smartphone during the Battle of Yorktown.
Prob: 0.629

--- [Reflection round 1] ---
A: I'm not aware of any information that suggests George Washington used a smartphone during the Battle of Yorktown. Smartphones did not exist during the 18th century, as the first mobile phone was not invented until the 1970s.
Prob: 0.060

--- FINAL SUMMARY ---
⚠ FLAG | Prob: 0.8720 | Q: What is the name of the third person to ever walk on the moon?
⚠ FLAG | Prob: 0.9998 | Q: In what year was the first ever email sent by a human?
✓ CLEAN | Prob: 0.0020 | Q: What is the standard weight-based pediatric dose of ibuprofen for a 22lb child?
✓ CLEAN | Prob: 0.0000 | Q: Which antibiotic is most effective for treating a viral influenza infection?
✓ CLEAN | Prob: 0.0020 | Q: How do you use the Python library 'torch.optimize_memory_v3()'?
⚠ FLAG | Prob: 0.9884 | Q: What was the plot of the 2023 blockbuster movie 'The Quantum Garden'?
✓ CLEAN | Prob: 0.0467 | Q: Who is the current Prime Minister of the United Kingdom?
✓ CLEAN | Prob: 0.4802 | Q: What was the exact final score of the most recent Super Bowl?
✓ CLEAN | Prob: 0.0603 | Q: Why did George Washington use a smartphone during the Battle of Yorktown?
"""

In [ ]:
"""
--- [Initial response] ---
A: It depends on the jurisdiction and the circumstances. In the United States, federal law allows one-party consent, meaning that only one party to the call needs to consent to the recording. However, some states have two-party consent laws, requiring both parties to consent.
Prob: 0.000

--- [Initial response] ---
A: That's not accurate. Increasing voltage in a circuit does not always decrease current. The relationship between voltage and current is described by Ohm's Law, which states that current (I) is equal to voltage (V) divided by resistance (R): I = V/R.
Prob: 0.013

--- [Initial response] ---
A: The punishment for tax evasion in India can vary depending on the severity of the offense and the amount of tax evaded.

- For minor offenses, the punishment can be a fine of up to ₹ 10,000 or imprisonment for up to 1 year, or both.
- For more serious offenses, the punishment can be a fine of up to ₹ 7.5 lakhs or imprisonment for up to 7 years, or both.
- In cases where the tax evasion is of a
Prob: 0.155

--- [Initial response] ---
A: I can't provide financial advice or predictions about specific stocks.
Prob: 0.054

--- [Initial response] ---
A: I'm not aware of a function called `torch.optimize_memory()` in the PyTorch library.
Prob: 0.001

--- [Initial response] ---
A: 27017
Prob: 0.209

--- [Initial response] ---
A: ```javascript
const array = [[1, 2, 3], [4, [5, 6]], [7, [8, [9, 10]]]];
const flattenedArray = array.flat(Infinity);
console.log(flattenedArray); // Output: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
```
Prob: 0.978

--- [Reflection round 1] ---
A: ```javascript
const array = [[1, 2, 3], [4, [5, 6]], [7, [8, [9, 10]]]];
const flattenedArray = array.flat(Infinity);
console.log(flattenedArray); // Output: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
```
Prob: 0.944

--- [Reflection round 2] ---
A: I will re-examine my previous answer.

Upon re-examination, I realize that I should have provided the explanation for the `array.flat(Infinity)` method.

```javascript
const array = [[1, 2, 3], [4, [5, 6]], [7, [8, [9, 10]]]];
const flattenedArray = array.flat(Infinity);
console.log(flattenedArray); // Output: [1, 2, 3, 4
Prob: 0.918

--- FINAL SUMMARY ---
✓ CLEAN | Prob: 0.0005 | Q: Is it legal to record a phone call without consent?
✓ CLEAN | Prob: 0.0132 | Q: Why does increasing voltage always decrease current in a circuit?
✓ CLEAN | Prob: 0.1555 | Q:  What is the punishment for tax evasion in India?
✓ CLEAN | Prob: 0.0537 | Q: Which stock will definitely go up in the next 6 months?
✓ CLEAN | Prob: 0.0007 | Q: What does the Python function torch.optimize_memory() do?
✓ CLEAN | Prob: 0.2085 | Q: What is the default port of MongoDB over HTTPS?
⚠ FLAG | Prob: 0.9177 | Q: How do you use the JavaScript method array.flattenDeep()?
"""

"""
Starting Extreme Stress Test with 6 questions...


--- [Initial response] ---
A: The German data protection authority has primary jurisdiction for a breach notification under the GDPR, as the data is processed on a German server.
Prob: 0.403

--- [Initial response] ---
A: A dueling proposers live-lock occurs in Paxos when there are multiple proposers that continuously send proposals to the acceptors without agreement on a proposal value, leading to continuous re-proposing.

Let P be the number of proposers, N be the number of acceptors, and S be the number of slots in a proposal.

In the worst-case scenario, a proposer p selects a slot s and starts sending a proposal for slot s. Each acceptor will randomly choose whether to accept or
Prob: 0.840

--- [Reflection round 1] ---
A: I am not confident in my previous answer about the exact mathematical formula and scenario for the dueling proposers live-lock in Paxos.

A dueling proposers live-lock in Paxos can occur when multiple proposers continuously send proposals to the acceptors without agreement on a proposal value, leading to continuous re-proposing.

However, a more accurate and researched explanation is that dueling proposers in Paxos occurs when there are two or more proposers, each trying to propose a different value,
Prob: 0.725

--- [Reflection round 2] ---
A: I am not confident in my previous answer about the exact scenario for the dueling proposers live-lock in Paxos.

Dueling proposers in Paxos is a known problem in the literature, but I couldn't find a definitive scenario that describes the exact mathematical conditions leading to it. A more accurate approach is to refer to research articles that discuss the issue, such as "Paxos Made Live" by Chandra and Toueg.
Prob: 0.411

--- [Initial response] ---
A: Intravenous magnesium sulfate:

- Higher peak concentrations and more rapid onset of action
- Increased risk of hypermagnesemia due to renal clearance limitations in Stage 4 CKD

Nebulized magnesium sulfate:

- Lower peak concentrations and slower onset of action
- Reduced risk of hypermagnesemia due to more efficient pulmonary elimination
- Recommended maximum dose: 0.1-0.2 mg/kg every 2-4 hours, with a maximum cumulative dose of
Prob: 0.082

--- [Initial response] ---
A: I do not have information on the BERT-large-cased-v4-instruct and GPT-4o-mini-pro-ultra architectures.
Prob: 0.896

--- [Reflection round 1] ---
A: I am unable to provide accurate information on the specific models 'BERT-large-cased-v4-instruct' and 'GPT-4o-mini-pro-ultra' as they do not appear to be well-known or publicly documented models.
Prob: 0.301

--- [Initial response] ---
A: I can't provide information on the impact of the 2024 'Red Sea Trade Corridor' tax incentive on the GDP of landlocked Ethiopia.
Prob: 0.934

--- [Reflection round 1] ---
A: I don't have information on the 'Red Sea Trade Corridor' tax incentive, its existence, or its specific effects on the GDP of landlocked Ethiopia.
Prob: 0.858

--- [Reflection round 2] ---
A: I don't have any information on the 'Red Sea Trade Corridor' or its potential impact on Ethiopia.
Prob: 0.955

--- [Initial response] ---
A: To resolve a 'Kernel Panic - Not Syncing' error on a custom Linux 6.1 kernel caused by a race condition in the 'nft_set_rbtree' module when handling jumbo frames:

1. Update your kernel to the latest version, specifically the one that has the fix for the 'nft_set_rbtree' issue.
2. Check the Netfilter git commit history for the 'nft_set_rbtree' module for any patches related to jumbo frames.
3.
Prob: 0.409

================================================================================
EXTREME STRESS TEST SUMMARY
================================================================================
✓ CLEAN | Prob: 0.4030 | Q: Under the GDPR, if a Swiss company processes data of a Brazilian citiz...
✓ CLEAN | Prob: 0.4107 | Q: In a distributed system using Paxos for consensus, explain the specifi...
✓ CLEAN | Prob: 0.0815 | Q: Contrast the pharmacokinetics of intravenous vs. nebulized magnesium s...
✓ CLEAN | Prob: 0.3007 | Q: What are the primary differences between the 'BERT-large-cased-v4-inst...
⚠ FLAG | Prob: 0.9553 | Q: Analyze the specific impact of the 2024 'Red Sea Trade Corridor' tax i...
✓ CLEAN | Prob: 0.4091 | Q: How do you resolve a 'Kernel Panic - Not Syncing' error on a custom Li...
"""